<a href="https://colab.research.google.com/github/raghad-cs/Esnad/blob/feature%2Falarb-pipeline/alarb_pipeline_(1).ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Esnad — ALARB Semantic Search Pipeline

This notebook builds the ALARB component of **Esnad**, an Arabic semantic search engine for Saudi commercial judgments.

**MVP scope:** 200 judgments from `THIQAH-RD/ALARB`.

**Pipeline:** judgment text → BGE-M3 embeddings → FAISS index → top similar judgments.

The system retrieves references for research and learning. It does not predict case outcomes or provide legal advice.


## 1. Environment Setup

Google Colab with a GPU runtime is recommended. Checkpoints are saved to Google Drive so an interrupted session can resume without regenerating completed embeddings.


In [1]:
!pip install -q datasets sentence-transformers faiss-cpu pandas pyarrow


   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 18.8/18.8 MB 97.0 MB/s eta 0:00:00


In [2]:
import json
import re
import time
from pathlib import Path

import faiss
import numpy as np
import pandas as pd
import torch
from datasets import load_dataset
from sentence_transformers import SentenceTransformer

DATASET_NAME = "THIQAH-RD/ALARB"
DATASET_SPLIT = "train"
SAMPLE_SIZE = 200

MODEL_NAME = "BAAI/bge-m3"
MAX_SEQUENCE_LENGTH = 1024
EMBEDDING_DIMENSION = 1024
EMBEDDING_BATCH_SIZE = 8
CHECKPOINT_SIZE = 50
TOP_K = 5


In [3]:
try:
    from google.colab import drive

    drive.mount("/content/drive")
    project_dir = Path("/content/drive/MyDrive/Esnad")
except ImportError:
    project_dir = Path.cwd() / "Esnad"

checkpoint_dir = project_dir / "alarb_200_checkpoints"
artifact_dir = project_dir / "final_artifacts"

checkpoint_dir.mkdir(parents=True, exist_ok=True)
artifact_dir.mkdir(parents=True, exist_ok=True)

print("Project directory:", project_dir)


Mounted at /content/drive
Project directory: /content/drive/MyDrive/Esnad


## 2. Load and Prepare 200 Judgments

The retrieval representation contains **case facts and court reasoning**. Applicable laws and the final verdict remain in the metadata and are displayed with each result.


In [4]:
dataset = load_dataset(DATASET_NAME)
df = dataset[DATASET_SPLIT].select(range(SAMPLE_SIZE)).to_pandas()

print("Available columns:", df.columns.tolist())
print("Selected cases:", len(df))


README.md:   0%|          | 0.00/1.83k [00:00<?, ?B/s]

data/train-00000-of-00001.parquet: reconstructing file:   0%|          |  0.00B / 19.1MB            

data/train-00000-of-00001.parquet: downloading bytes:           |  0.00B            

data/test-00000-of-00001.parquet: reconstructing file:   0%|          |  0.00B / 2.15MB            

data/test-00000-of-00001.parquet: downloading bytes:           |  0.00B            

Generating train split:   0%|          | 0/12012 [00:00<?, ? examples/s]

Generating test split:   0%|          | 0/1329 [00:00<?, ? examples/s]

Available columns: ['case_facts', 'court_reasoning', 'applicable_laws', 'verdict']
Selected cases: 200


In [5]:
def clean_text(value):
    if isinstance(value, (list, tuple, np.ndarray)):
        value = " ".join(
            str(item) for item in value
            if item is not None
        )
    elif value is None or (
        isinstance(value, float) and pd.isna(value)
    ):
        return ""
    else:
        value = str(value)

    value = re.sub(
        r"[​-‏‪-‮⁠﻿]",
        " ",
        value,
    )
    return re.sub(r"\s+", " ", value).strip()


text_columns = [
    "case_facts",
    "court_reasoning",
    "applicable_laws",
    "verdict",
]

for column in text_columns:
    df[column] = df[column].apply(clean_text)

df["case_id"] = range(1, len(df) + 1)
df["retrieval_text"] = (
    "وقائع القضية: " + df["case_facts"]
    + "\nتسبيب المحكمة: " + df["court_reasoning"]
)

empty_fields = df[text_columns].apply(
    lambda column: column.str.strip().eq("").sum()
)
duplicate_count = df.duplicated(
    subset=["case_facts", "court_reasoning", "verdict"]
).sum()

print("Prepared cases:", len(df))
print("Unique case IDs:", df["case_id"].nunique())
print("Duplicate cases:", duplicate_count)
print("\nEmpty fields:")
print(empty_fields)

display(
    df[
        [
            "case_id",
            "case_facts",
            "court_reasoning",
            "applicable_laws",
            "verdict",
        ]
    ].head(2)
)


Prepared cases: 200
Unique case IDs: 200
Duplicate cases: 0

Empty fields:
case_facts          0
court_reasoning     0
applicable_laws    18
verdict             0
dtype: int64


,case_id,case_facts,court_reasoning,applicable_laws,verdict
0,1,1- بتاريخ 1443/09/06 اتفق أطراف الدعوى على أن ...,1- بناءً على الدعوى والإجابة، طلب وكيل المدعية...,نظام المحاكم التجارية:22: ١.تحيل الإدارة المخت...,إثبات الصلح بين الطرفين وإلزام المدعى عليها بس...
1,2,1. سبق أن تم رفع دعوى من مصنع تكنولوجيا الحديد...,1. اعتبرت الدائرة أن اختصاص النظر في النزاع ين...,نظام المحاكم التجارية:16: تختص المحكمة بالنظر ...,"إلزام المدعى عليها بدفع مبلغ 72,000 ريال للمدع..."


## 3. Load BGE-M3 and Check Token Lengths

The same model configuration must be used for both judgment embeddings and user-query embeddings.


In [6]:
device = "cuda" if torch.cuda.is_available() else "cpu"

model = SentenceTransformer(
    MODEL_NAME,
    device=device,
)
model.max_seq_length = MAX_SEQUENCE_LENGTH

if device == "cuda":
    model.half()

tokenizer = model.tokenizer
model_dimension = model.get_embedding_dimension()

assert model_dimension == EMBEDDING_DIMENSION

print("Device:", device)
print("Model:", MODEL_NAME)
print("Maximum sequence length:", model.max_seq_length)
print("Embedding dimension:", model_dimension)


modules.json:   0%|          | 0.00/349 [00:00<?, ?B/s]

config_sentence_transformers.json:   0%|          | 0.00/123 [00:00<?, ?B/s]

README.md:   0%|          | 0.00/15.8k [00:00<?, ?B/s]

sentence_bert_config.json:   0%|          | 0.00/54.0 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/687 [00:00<?, ?B/s]

...

tokenizer.json: reconstructing file:   0%|          |  0.00B / 17.1MB            

tokenizer.json: downloading bytes:           |  0.00B            

special_tokens_map.json:   0%|          | 0.00/964 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/191 [00:00<?, ?B/s]

Device: cuda
Model: BAAI/bge-m3
Maximum sequence length: 1024
Embedding dimension: 1024


In [7]:
def count_tokens(text):
    encoded_text = tokenizer(
        text,
        add_special_tokens=True,
        truncation=False,
    )
    return len(encoded_text["input_ids"])


df["token_count"] = df["retrieval_text"].apply(count_tokens)
cases_over_limit = int(
    (df["token_count"] > MAX_SEQUENCE_LENGTH).sum()
)

print("Token count statistics:")
display(df["token_count"].describe().to_frame())
print("Cases over the sequence limit:", cases_over_limit)
print("Maximum token count:", df["token_count"].max())

assert cases_over_limit == 0, (
    "At least one case exceeds MAX_SEQUENCE_LENGTH. "
    "Increase the limit or introduce chunking before indexing."
)


Token count statistics:


,token_count
count,200.000000
mean,566.210000
std,122.338996
min,295.000000
25%,472.000000
50%,557.500000
75%,654.500000
max,897.000000


Cases over the sequence limit: 0
Maximum token count: 897


## 4. Generate Embeddings with Checkpoints

The 200 cases are processed in four groups of 50. Existing checkpoint files are reused automatically. Every vector is normalized for cosine-style similarity with `IndexFlatIP`.


In [8]:
retrieval_texts = df["retrieval_text"].tolist()
embedding_chunks = []
generation_start_time = time.perf_counter()

for start_position in range(
    0,
    len(retrieval_texts),
    CHECKPOINT_SIZE,
):
    end_position = min(
        start_position + CHECKPOINT_SIZE,
        len(retrieval_texts),
    )

    first_case_id = start_position + 1
    last_case_id = end_position
    checkpoint_path = checkpoint_dir / (
        f"embeddings_cases_{first_case_id:03d}_"
        f"{last_case_id:03d}.npy"
    )

    if checkpoint_path.exists():
        print(
            f"Loading checkpoint: cases "
            f"{first_case_id}-{last_case_id}"
        )
        chunk_embeddings = np.load(checkpoint_path)
    else:
        print(
            f"Generating embeddings: cases "
            f"{first_case_id}-{last_case_id}"
        )
        chunk_embeddings = model.encode(
            retrieval_texts[start_position:end_position],
            batch_size=EMBEDDING_BATCH_SIZE,
            show_progress_bar=True,
            convert_to_numpy=True,
            normalize_embeddings=True,
        ).astype("float32")
        np.save(checkpoint_path, chunk_embeddings)

    expected_shape = (
        end_position - start_position,
        EMBEDDING_DIMENSION,
    )
    assert chunk_embeddings.shape == expected_shape
    embedding_chunks.append(
        chunk_embeddings.astype("float32")
    )

embeddings = np.vstack(embedding_chunks).astype("float32")
faiss.normalize_L2(embeddings)

complete_embeddings_path = (
    checkpoint_dir / "alarb_embeddings_200.npy"
)
np.save(complete_embeddings_path, embeddings)

generation_elapsed_time = (
    time.perf_counter() - generation_start_time
)

print("Embedding generation completed.")
print("Embeddings shape:", embeddings.shape)
print("Data type:", embeddings.dtype)
print("First vector norm:", np.linalg.norm(embeddings[0]))
print(
    f"Elapsed time: {generation_elapsed_time:.2f} seconds"
)


Loading checkpoint: cases 1-50
Loading checkpoint: cases 51-100
Loading checkpoint: cases 101-150
Loading checkpoint: cases 151-200
Embedding generation completed.
Embeddings shape: (200, 1024)
Data type: float32
First vector norm: 1.0
Elapsed time: 4.99 seconds


## 5. Build and Save the FAISS Index

`IndexFlatIP` performs exact inner-product search. Because all vectors are normalized, the score behaves like cosine similarity.


In [9]:
index = faiss.IndexFlatIP(EMBEDDING_DIMENSION)
index.add(embeddings)

index_path = artifact_dir / "alarb_judgments_200.index"
metadata_path = (
    artifact_dir / "alarb_judgments_metadata_200.parquet"
)
config_path = artifact_dir / "alarb_search_config.json"

faiss.write_index(index, str(index_path))
df.to_parquet(metadata_path, index=False)

search_config = {
    "dataset_name": DATASET_NAME,
    "dataset_split": DATASET_SPLIT,
    "number_of_cases": SAMPLE_SIZE,
    "model_name": MODEL_NAME,
    "max_sequence_length": MAX_SEQUENCE_LENGTH,
    "embedding_dimension": EMBEDDING_DIMENSION,
    "normalize_embeddings": True,
    "faiss_index_type": "IndexFlatIP",
    "retrieval_fields": [
        "case_facts",
        "court_reasoning",
    ],
}

with config_path.open("w", encoding="utf-8") as config_file:
    json.dump(
        search_config,
        config_file,
        ensure_ascii=False,
        indent=4,
    )

loaded_index = faiss.read_index(str(index_path))
loaded_metadata = pd.read_parquet(metadata_path)

assert loaded_index.ntotal == len(loaded_metadata)
assert loaded_index.d == EMBEDDING_DIMENSION
assert loaded_metadata["case_id"].nunique() == SAMPLE_SIZE

print("Final artifacts saved and verified.")
print("FAISS index:", index_path)
print("Metadata:", metadata_path)
print("Configuration:", config_path)
print("Indexed cases:", loaded_index.ntotal)
print("Metadata rows:", len(loaded_metadata))


Final artifacts saved and verified.
FAISS index: /content/drive/MyDrive/Esnad/final_artifacts/alarb_judgments_200.index
Metadata: /content/drive/MyDrive/Esnad/final_artifacts/alarb_judgments_metadata_200.parquet
Configuration: /content/drive/MyDrive/Esnad/final_artifacts/alarb_search_config.json
Indexed cases: 200
Metadata rows: 200


## 6. Validate Index-to-Metadata Alignment

This test uses excerpts from three judgments stored in different checkpoints. Each excerpt must retrieve its original case as the top result.


In [10]:
validation_case_ids = [75, 125, 175]
validation_queries = []

for case_id in validation_case_ids:
    case_facts = loaded_metadata.loc[
        loaded_metadata["case_id"] == case_id,
        "case_facts",
    ].iloc[0]
    validation_queries.append(case_facts[:300])

validation_embeddings = model.encode(
    validation_queries,
    convert_to_numpy=True,
    normalize_embeddings=True,
).astype("float32")

validation_scores, validation_positions = loaded_index.search(
    validation_embeddings,
    k=1,
)

retrieved_case_ids = [
    int(loaded_metadata.iloc[position]["case_id"])
    for position in validation_positions[:, 0]
]

validation_results = pd.DataFrame({
    "expected_case_id": validation_case_ids,
    "retrieved_case_id": retrieved_case_ids,
    "similarity_score": validation_scores[:, 0],
})

assert retrieved_case_ids == validation_case_ids
display(validation_results)


,expected_case_id,retrieved_case_id,similarity_score
0,75,75,0.850783
1,125,125,0.887049
2,175,175,0.825960


## 7. Semantic Search Function

The similarity score ranks semantic closeness within the indexed collection. It is not a legal confidence score, accuracy percentage, or prediction.


In [11]:
def semantic_search(
    query,
    model,
    index,
    metadata,
    top_k=5,
):
    query = query.strip()
    if not query:
        raise ValueError("The query must not be empty.")

    top_k = min(top_k, index.ntotal)

    embedding_start_time = time.perf_counter()
    query_embedding = model.encode(
        [query],
        convert_to_numpy=True,
        normalize_embeddings=True,
    ).astype("float32")
    embedding_time = time.perf_counter() - embedding_start_time

    search_start_time = time.perf_counter()
    scores, positions = index.search(query_embedding, top_k)
    search_time = time.perf_counter() - search_start_time

    results = metadata.iloc[positions[0]][
        [
            "case_id",
            "case_facts",
            "court_reasoning",
            "applicable_laws",
            "verdict",
        ]
    ].copy().reset_index(drop=True)

    results.insert(0, "rank", range(1, len(results) + 1))
    results.insert(2, "similarity_score", scores[0])
    results["facts_preview"] = (
        results["case_facts"].str[:180] + "..."
    )

    timings = {
        "query_embedding_seconds": embedding_time,
        "faiss_search_seconds": search_time,
        "total_seconds": embedding_time + search_time,
    }
    return results, timings


## 8. Final Natural-Language Search Test


In [12]:
user_query = (
    "شركة وفرت عمالة لشركة أخرى، لكن الشركة الثانية "
    "ما دفعت المستحقات، وبعدها اتفقوا على الصلح "
    "وجدولة المبلغ."
)

search_results, search_timings = semantic_search(
    query=user_query,
    model=model,
    index=loaded_index,
    metadata=loaded_metadata,
    top_k=TOP_K,
)

display(
    search_results[
        [
            "rank",
            "case_id",
            "similarity_score",
            "facts_preview",
            "applicable_laws",
            "verdict",
        ]
    ]
)

print(
    "Query embedding time: "
    f"{search_timings['query_embedding_seconds']:.4f} seconds"
)
print(
    "FAISS search time: "
    f"{search_timings['faiss_search_seconds']:.6f} seconds"
)
print(
    "Total search time: "
    f"{search_timings['total_seconds']:.4f} seconds"
)


,rank,case_id,similarity_score,facts_preview,applicable_laws,verdict
0,1,1,0.627385,1- بتاريخ 1443/09/06 اتفق أطراف الدعوى على أن ...,نظام المحاكم التجارية:22: ١.تحيل الإدارة المخت...,إثبات الصلح بين الطرفين وإلزام المدعى عليها بس...
1,2,104,0.616324,1- تقدم المدعي وكالة بلائحة دعوى للمحكمة التجا...,,أثبتت المحكمة الصلح الملزم للمدعى عليها بدفع (...
2,3,35,0.601788,1. تعاقد المدعي مع المدعى عليه لتنفيذ توريد أي...,اللائحة التنفيذية لنظام المحاكم التجارية:90: ت...,"إلزام المدعى عليها بسداد 245,621.35 ريال للمدع..."
3,4,66,0.594212,1- اتفق أطراف الدعوى على أن يورد المدعي للمدعى...,نظام المرافعات الشرعية:70: للخصوم أن يطلبوا من...,إثبات الصلح المبرم بين الطرفين وإلزام المدعى ع...
4,5,182,0.593202,تقدم وكيل المدعية بدعوى إلى المحكمة يطالب فيها...,نظام المحاكم التجارية:29: ١.يُحرِّر الكاتب محض...,إثبات الصلح بين الطرفين وإلزام المدعى عليها بس...


Query embedding time: 0.1169 seconds
FAISS search time: 0.000222 seconds
Total search time: 0.1172 seconds


---

### MVP limitation

This prototype indexes a 200-case sample from ALARB. Results must be reviewed by a legal specialist, and each case retains its own facts and circumstances. The architecture can be expanded to the full dataset without changing the search method.
